# 02. 공통 VGGFace2 부모 1개 생성

## 1. 설정
공통 변경은 00, 이 단계만 변경하려면 아래 변수를 편집하세요.

In [ ]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])
HEADS = list(saved["campaign"]["heads"])
PARENT_SEED = saved["campaign"]["parent_seed"]
PRETRAIN_EPOCHS = saved["campaign"]["pretrain_epochs"]
PARENT_CHECKPOINT = saved["campaign"]["parent_checkpoint"]
PARENT_SHA256 = saved["campaign"]["parent_sha256"]
FINETUNE_DATASETS = list(saved["campaign"]["finetune_datasets"])
FINETUNE_PROFILES = saved["finetune_profiles"]
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]
NUM_WORKERS = saved["training"]["num_workers"]
PREFETCH_FACTOR = saved["training"]["prefetch_factor"]
PERSISTENT_WORKERS = saved["training"]["persistent_workers"]
TORCH_THREADS = saved["training"]["torch_threads"]
LOSS_SYNC_BATCHES = saved["training"].get("loss_sync_batches", 20)
RECOVERY_MINUTES = saved["training"]["recovery_minutes"]
DEVICE = saved["project"]["device"]
DATASET = dict(saved["dataset"])  # 02는 VGGFace2, 03은 선택한 대상 profile을 사용
BACKBONE = dict(saved["backbone"])
TRAINING = dict(saved["training"])
HEAD_PARAMETERS = saved["heads"]
PATHS = dict(saved["paths"])
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]
EXECUTE = True  # 학습하려면 명시적으로 True 설정
STAGE = "pretrain"
RUN_SMOKE_TEST = False
RESUME_CHECKPOINTS = {"pretrain_arcface_seed8972": 'F:\\FaceX-Zoo\\fine-tune\\checkpoints\\shared_parent_v2\\pretrain_arcface_seed8972\\pretrain_mobilefacenet_1024d_recovery_e1_b3102_196a5044.pt'}  # run_id -> 정확한 재개 checkpoint 경로
RUN_BENCHMARK = False  # True면 속도 비교만 실행하고 정식 학습은 건너뜀
WORKER_CANDIDATES = [0,2,4]
BENCHMARK_STEPS = 100
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
 test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
 meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS})


## 2. 계획

In [ ]:
plan = plan_matrix(settings, STAGE)
for row in plan:
    print(row)
print("예정:",len(plan),"EXECUTE:",EXECUTE,"SMOKE:",RUN_SMOKE_TEST)

## 3. 재개 정책

In [ ]:
print("공통 부모:", PARENT_CHECKPOINT or "02의 PARENT_SEED 고정 경로", "parent seed:", PARENT_SEED)
print("시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.")
from modules.process_runner import launch_campaign
if RUN_BENCHMARK:
    benchmark_results = launch_campaign(BASE_DIR,settings,"benchmark",benchmark_workers=WORKER_CANDIDATES,benchmark_steps=BENCHMARK_STEPS)
    print(benchmark_results)

## 4. 별도 프로세스 실행
노트북 Interrupt는 stop 파일을 만들고 학습은 optimizer step 완료 후 재개 checkpoint를 저장합니다. Kernel 강제 종료는 이 절차를 실행하지 않을 수 있으므로 출력된 stop 파일 경로 또는 progress.json을 확인하세요. 입력 처리만 최적화하며 baseline batch 64·Adam·40 epoch는 유지합니다.

In [ ]:
from modules.process_runner import launch_campaign
results = []
if EXECUTE and not RUN_BENCHMARK:
    results=launch_campaign(BASE_DIR,settings,STAGE,RESUME_CHECKPOINTS,smoke=RUN_SMOKE_TEST)
else:
    print("계획 또는 benchmark 모드: 정식 작업 미실행")

## 5. 완료 상태

In [ ]:
for result in results:
    print(result)
print("정식 완료:",sum(r.get("completed") is True for r in results),"/",len(plan))
print("실패 시 summary에 미실행 범위를 기록하고 후속 작업을 중단합니다.")
print("해석 안내:",BASE_DIR/"fine-tune/README.md")